In [1]:
pip install pandas numpy matplotlib seaborn tqdm IPython pyarrow fastparquet torch


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Some common libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dataset preparation

In [3]:
# Load dataset
df = pd.read_csv('dataset/train.csv')
print(f"Initial dataset shape: {df.shape}")
display(df.head())

print("\nMissing values per column:")
print(df.isna().sum())

Initial dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [4]:
def reformat_name(name):
    if ',' in name or '.' in name:
        title = name.split(', ')[1].split('. ')[0].strip()
        return f"{title}"
    return name
    
def reformat_sex(sex):
    return 1 if sex =='male' else 0

def reformat_embark(embark):
    if embark == 'S':
        return 0
    elif embark == 'C':
        return 1
    elif embark == 'Q':
        return 2

In [5]:
df['Name'] = df['Name'].apply(reformat_name)
df['Sex'] = df['Sex'].apply(reformat_sex)
df['Embarked'] = df['Embarked'].apply(reformat_embark)

In [6]:
for col in df.columns:
    num_unique_values = df[col].nunique()
    if num_unique_values / len(df) > 0.5:
        print(f"Column '{col}' has high cardinality with {num_unique_values} unique values over {len(df)} data.")

Column 'PassengerId' has high cardinality with 891 unique values over 891 data.
Column 'Ticket' has high cardinality with 681 unique values over 891 data.


In [7]:
IGNORED_COLUMNS = ['PassengerId', 'Ticket', 'Cabin']

df.drop(columns=IGNORED_COLUMNS, inplace=True, axis=1)
print(f"\nTruncated dataset shape: {df.shape}")
display(df.head(n=10))


Truncated dataset shape: (891, 9)


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,Mr,1,22.0,1,0,7.2500,0.0
1,1,1,Mrs,0,38.0,1,0,71.2833,1.0
2,1,3,Miss,0,26.0,0,0,7.9250,0.0
3,1,1,Mrs,0,35.0,1,0,53.1000,0.0
4,0,3,Mr,1,35.0,0,0,8.0500,0.0
5,0,3,Mr,1,NaN,0,0,8.4583,2.0
6,0,1,Mr,1,54.0,0,0,51.8625,0.0
7,0,3,Master,1,2.0,3,1,21.0750,0.0
8,1,3,Mrs,0,27.0,0,2,11.1333,0.0
9,1,2,Mrs,0,14.0,1,0,30.0708,1.0


In [8]:
# Filled dataset
for col in df.select_dtypes(include=[np.number]).columns:
    skewness = df[col].skew()
    print(f"{col} → skewness: {skewness:.2f}")
    
    if abs(skewness) < 0.5:
        fill_value = df[col].mean()
        method = "mean"
    else:
        fill_value = df[col].median()
        method = "median"
    if df[col].isnull().sum() > 0:
        df.fillna({col: fill_value}, inplace=True)
        print(f"Filled {col} using {method} ({fill_value:.2f})")

Survived → skewness: 0.48
Pclass → skewness: -0.63
Sex → skewness: -0.62
Age → skewness: 0.39
Filled Age using mean (29.70)
SibSp → skewness: 3.70
Parch → skewness: 2.75
Fare → skewness: 4.79
Embarked → skewness: 1.54
Filled Embarked using median (0.00)


In [9]:
df_encoded = pd.get_dummies(df, columns=['Pclass', 'Name', 'Sex', 'Embarked'], dtype=int, drop_first=True)
df_encoded

,Survived,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Name_Col,Name_Don,Name_Dr,...,Name_Mme,Name_Mr,Name_Mrs,Name_Ms,Name_Rev,Name_Sir,Name_the Countess,Sex_1,Embarked_1.0,Embarked_2.0
0,0,22.000000,1,0,7.2500,0,1,0,0,0,...,0,1,0,0,0,0,0,1,0,0
1,1,38.000000,1,0,71.2833,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
2,1,26.000000,0,0,7.9250,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,35.000000,1,0,53.1000,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,35.000000,0,0,8.0500,0,1,0,0,0,...,0,1,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,27.000000,0,0,13.0000,1,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
887,1,19.000000,0,0,30.0000,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
888,0,29.699118,1,2,23.4500,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
889,1,26.000000,0,0,30.0000,0,0,0,0,0,...,0,1,0,0,0,0,0,1,1,0


## Features exploration

In [10]:
for col in df.columns:
    if col == 'Survived':
        continue
    for unique_value in df[col].unique():
        if pd.isna(unique_value):
            continue
        count = df.loc[df[col] == unique_value]["Survived"]
        print(f"% of {col} {unique_value} who survived:", sum(count)/len(count))
    print("-----\n")

% of Pclass 3 who survived: 0.24236252545824846
% of Pclass 1 who survived: 0.6296296296296297
% of Pclass 2 who survived: 0.47282608695652173
-----

% of Name Mr who survived: 0.15667311411992263
% of Name Mrs who survived: 0.792
% of Name Miss who survived: 0.6978021978021978
% of Name Master who survived: 0.575
% of Name Don who survived: 0.0
% of Name Rev who survived: 0.0
% of Name Dr who survived: 0.42857142857142855
% of Name Mme who survived: 1.0
% of Name Ms who survived: 1.0
% of Name Major who survived: 0.5
% of Name Lady who survived: 1.0
% of Name Sir who survived: 1.0
% of Name Mlle who survived: 1.0
% of Name Col who survived: 0.5
% of Name Capt who survived: 0.0
% of Name the Countess who survived: 1.0
% of Name Jonkheer who survived: 0.0
-----



% of Sex 1 who survived: 0.18890814558058924
% of Sex 0 who survived: 0.7420382165605095
-----

% of Age 22.0 who survived: 0.4074074074074074
% of Age 38.0 who survived: 0.45454545454545453
% of Age 26.0 who survived: 0.3333333333333333
% of Age 35.0 who survived: 0.6111111111111112
% of Age 29.69911764705882 who survived: 0.2937853107344633
% of Age 54.0 who survived: 0.375
% of Age 2.0 who survived: 0.3
% of Age 27.0 who survived: 0.6111111111111112
% of Age 14.0 who survived: 0.5
% of Age 4.0 who survived: 0.7
% of Age 58.0 who survived: 0.6
% of Age 20.0 who survived: 0.2
% of Age 39.0 who survived: 0.35714285714285715
% of Age 55.0 who survived: 0.5
% of Age 31.0 who survived: 0.47058823529411764
% of Age 34.0 who survived: 0.4
% of Age 15.0 who survived: 0.8
% of Age 28.0 who survived: 0.28
% of Age 8.0 who survived: 0.5
% of Age 19.0 who survived: 0.36
% of Age 40.0 who survived: 0.46153846153846156
% of Age 66.0 who survived: 0.0
% of Age 42.0 who survived: 0.46153846153846156

## Data loaders

In [11]:
import torch
from torch.utils.data import DataLoader, TensorDataset

class DataSet(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
dataset = TensorDataset(
    torch.tensor(df_encoded.drop(columns=['Survived']).values),
    torch.tensor(df_encoded['Survived'].values)
)
# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

In [12]:
# Retrieve and print a single random batch
X, y = next(iter(dataloader))
print("Data shape:", X.shape)
print("Labels:", y)

Data shape: torch.Size([16, 25])
Labels: tensor([0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0])


In [13]:
for batch_idx, (data, target) in enumerate(dataloader):
    print(f"Batch {batch_idx+1}")
    print(f"  Data shape: {data.shape}")     # Expect [batch_size, 10]
    print(f"  Target shape: {target.shape}") # Expect [batch_size]
    print(f"  Data type: {data}")
    print(f"  Target type: {target}")
    
    # Break after first batch for quick test
    break

Batch 1
  Data shape: torch.Size([16, 25])
  Target shape: torch.Size([16])
  Data type: tensor([[50.0000,  0.0000,  0.0000, 10.5000,  1.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000],
        [42.0000,  0.0000,  0.0000, 13.0000,  1.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  0.0000,  1.0000,  0.0000,
          0.0000],
        [25.0000,  0.0000,  0.0000,  7.7417,  0.0000,  1.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,
          1.0000],
        [17.0000,  0.0000,  0.0000, 10.5000,  1.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,

## Model experiments

### Random Forest

### KNN

### NN

## Training

In [14]:
import torch.nn as nn
import torch.optim as optim

class Trainer:
    def __init__(self, model, optimizer, criterion,  lr=1e-3):
        self.model = model
        self.optimizer = optimizer(self.model.parameters(), lr=lr)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def train_one_epoch(self):
        self.model.train()
        running_loss = 0.0
        for batch in self.train_loader:
            inputs, labels = [bat.to(self.device) for bat in batch]
            self.optimizer.zero_grad()
            
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item()
        epoch_loss = running_loss / len(self.train_loader)
        return epoch_loss

    def evaluate(self):
        train_score = self.model.score(self.X_train, self.y_train)
        val_score = self.model.score(self.X_val, self.y_val)
        print(f"Train Score: {train_score:.4f}")
        print(f"Validation Score: {val_score:.4f}")